# XGboost prediction model

In [24]:
# Import necessary libraries and data
import pandas as pd
import re
import numpy as np
import os
%run 2.3_xgboost_dataloader_companies.py


Loaded: data\final_panel.csv

  final_panel
  Rows: 27,965    Columns: 21

|   # | Column                            | Dtype   |   Non-Null |   Null |
|-----|-----------------------------------|---------|------------|--------|
|   0 | Company name Latin alphabet       | str     |     27,965 |      0 |
|   1 | Year of incorporation             | int64   |     27,965 |      0 |
|   2 | Company Age                       | int64   |     27,965 |      0 |
|   3 | Bankrupt                          | int64   |     27,965 |      0 |
|   4 | NACE Section                      | str     |     27,965 |      0 |
|   5 | NACE Rev. 2, core code (4 digits) | int64   |     27,965 |      0 |
|   6 | Year                              | int64   |     27,965 |      0 |
|   7 | Current assets                    | float64 |     27,941 |     24 |
|   8 | Current liabilities               | float64 |     27,922 |     43 |
|   9 | P/L before tax                    | float64 |     27,930 |     35 |
|  10 | Share

In [27]:
# Observed universe counts from Moody's export
universe_total    = 64_052
universe_bankrupt =  1_136
universe_active   = universe_total - universe_bankrupt

# Sample counts
sample_active   = len(df_final_panel[df_final_panel["Bankrupt"] == 0]
                      ["Company name Latin alphabet"].unique())
sample_bankrupt = len(df_final_panel[df_final_panel["Bankrupt"] == 1]
                      ["Company name Latin alphabet"].unique())

# Compute weights
spw_sample   = sample_active   / sample_bankrupt
spw_universe = universe_active / universe_bankrupt

print(f"Universe counts (Moody's filtered search):")
print(f"  Total:                {universe_total:,}")
print(f"  Active (derived):     {universe_active:,}")
print(f"  Bankrupt:             {universe_bankrupt:,}")
print(f"  Bankruptcy rate:      {universe_bankrupt/universe_total:.2%}")

print(f"\nSample counts:")
print(f"  Active companies:     {sample_active:,}")
print(f"  Bankrupt companies:   {sample_bankrupt:,}")
print(f"  Bankruptcy rate:      {sample_bankrupt/(sample_active+sample_bankrupt):.2%}")
print(f"  Oversampling factor:  {(sample_bankrupt/(sample_active+sample_bankrupt)) / (universe_bankrupt/universe_total):.1f}×")

print(f"\nscale_pos_weight options:")
print(f"  Sample ratio:         {spw_sample:.1f}  (performs well on test set)")
print(f"  Universe ratio:       {spw_universe:.1f}  (calibrated to Finnish SME universe)")
print(f"\n→ Using universe ratio: {spw_universe:.1f}")

scale_pos_weight = spw_universe


# Split must be by company — not by row — to prevent leakage.
# The same company's rows must appear entirely in train or
# entirely in test, never in both.
#
# Stratification ensures the ~11% bankruptcy rate is preserved
# in both train and test splits.
#
# Split: 80% train / 20% test

from sklearn.model_selection import train_test_split

company_labels = (
    df_final_panel
    .groupby("Company name Latin alphabet")["Bankrupt"]
    .first()
    .reset_index()
)

train_companies, test_companies = train_test_split(
    company_labels["Company name Latin alphabet"],
    test_size=0.2,
    random_state=42,
    stratify=company_labels["Bankrupt"]
)

train_companies = set(train_companies)
test_companies  = set(test_companies)

df_train = df_final_panel[
    df_final_panel["Company name Latin alphabet"].isin(train_companies)
].copy().reset_index(drop=True)

df_test = df_final_panel[
    df_final_panel["Company name Latin alphabet"].isin(test_companies)
].copy().reset_index(drop=True)

print(f"\nTrain set:")
print(f"  Rows:                 {len(df_train):,}")
print(f"  Unique companies:     {df_train['Company name Latin alphabet'].nunique():,}")
print(f"  Bankrupt companies:   {(df_train.groupby('Company name Latin alphabet')['Bankrupt'].first()==1).sum():,}")
print(f"  Active companies:     {(df_train.groupby('Company name Latin alphabet')['Bankrupt'].first()==0).sum():,}")

print(f"\nTest set:")
print(f"  Rows:                 {len(df_test):,}")
print(f"  Unique companies:     {df_test['Company name Latin alphabet'].nunique():,}")
print(f"  Bankrupt companies:   {(df_test.groupby('Company name Latin alphabet')['Bankrupt'].first()==1).sum():,}")
print(f"  Active companies:     {(df_test.groupby('Company name Latin alphabet')['Bankrupt'].first()==0).sum():,}")

overlap = train_companies & test_companies
print(f"\nCompany overlap between train and test: {len(overlap)} (must be 0)")

Universe counts (Moody's filtered search):
  Total:                64,052
  Active (derived):     62,916
  Bankrupt:             1,136
  Bankruptcy rate:      1.77%

Sample counts:
  Active companies:     8,993
  Bankrupt companies:   1,129
  Bankruptcy rate:      11.15%
  Oversampling factor:  6.3×

scale_pos_weight options:
  Sample ratio:         8.0  (performs well on test set)
  Universe ratio:       55.4  (calibrated to Finnish SME universe)

→ Using universe ratio: 55.4

Train set:
  Rows:                 22,314
  Unique companies:     8,097
  Bankrupt companies:   903
  Active companies:     7,194

Test set:
  Rows:                 5,651
  Unique companies:     2,025
  Bankrupt companies:   226
  Active companies:     1,799

Company overlap between train and test: 0 (must be 0)
